In [ ]:
# Download the OpenCV Haar Cascade for face detection
!wget https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml

# All necessary imports
import cv2
from google.colab import drive, output
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript
from base64 import b64decode
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img

print("✅ Setup complete.")

--2025-11-15 16:12:18--  https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 930127 (908K) [text/plain]
Saving to: ‘haarcascade_frontalface_default.xml’

haarcascade_frontal 100%[===================>] 908.33K  --.-KB/s    in 0.03s   

2025-11-15 16:12:18 (30.7 MB/s) - ‘haarcascade_frontalface_default.xml’ saved [930127/930127]

✅ Setup complete.


In [ ]:
# Function to take a photo from the webcam
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture button to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = output.eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
# --- 1. Define Key Variables ---
IMG_SIZE = 100
# This MUST match the alphabetical order your original LabelEncoder learned
label_map = ['anger', 'disgust', 'fear', 'happiness', 'neutrality', 'sadness', 'surprise']

# --- 2. Define the File-Prediction Helper Function (Optional) ---
# This is your 'ef' function, useful if you want to test files
def ef(image_path):
    img = load_img(image_path, color_mode='grayscale', target_size=(IMG_SIZE, IMG_SIZE))
    feature = np.array(img)
    feature = feature.reshape(1, IMG_SIZE, IMG_SIZE, 1)
    return feature / 255.0

# --- 3. Mount Drive and Load Model ---
print("Mounting Google Drive...")
drive.mount('/content/drive')

model_path = '/content/drive/MyDrive/raf_db_model.keras'
print(f"Loading model from {model_path}...")
try:
    model = load_model(model_path)
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("Please make sure 'raf_db_model.keras' is in your Google Drive's main folder.")

Mounting Google Drive...
Mounted at /content/drive
Loading model from /content/drive/MyDrive/raf_db_model.keras...
✅ Model loaded successfully!


In [ ]:
# --- 4. Load the Face Detector ---
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

# --- 5. Run the Prediction Demo ---
print("\nClick 'Capture' in the output below. You may need to grant camera permissions.")
try:
    filename = take_photo()
    frame = cv2.imread(filename)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    if len(faces) == 0:
        print("No faces detected.")
    else:
        print(f"Detected {len(faces)} face(s).")

    # Loop over each detected face
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]

        # Preprocess the cropped face for your model
        final_face = cv2.resize(roi_gray, (IMG_SIZE, IMG_SIZE))
        final_face = final_face.reshape(1, IMG_SIZE, IMG_SIZE, 1)
        final_face = final_face / 255.0

        # Make the prediction
        pred = model.predict(final_face, verbose=0)
        pred_label = label_map[pred.argmax()]

        # Draw the result on the original color image
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, pred_label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Display the final image
    cv2_imshow(frame)

except Exception as err:
    print(f"An error occurred: {err}")


Click 'Capture' in the output below. You may need to grant camera permissions.


<IPython.core.display.Javascript object>

An error occurred: NotAllowedError: Permission denied
